<a href="https://colab.research.google.com/github/Akshatha7710/smart-tea-estate-management-system/blob/soil_quality-analysis_and_predictive_modeling/Soil_Quality_Analysis_and_Predictive_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt

# ===============================
# 1️⃣ PREPARE TIME INDEX
# ===============================

clean_df["Date"] = pd.to_datetime(clean_df[["Year", "Month"]].assign(DAY=1))
clean_df = clean_df.sort_values("Date")
clean_df = clean_df.set_index("Date")

# ===============================
# 2️⃣ FEATURE ENGINEERING
# ===============================

# Lag Features
clean_df["Rain_lag1"] = clean_df["Rainfall"].shift(1)
clean_df["Wet_lag1"] = clean_df["WetDays"].shift(1)

# Rolling Features
clean_df["Rain_roll3"] = clean_df["Rainfall"].rolling(3).mean()
clean_df["Wet_roll3"] = clean_df["WetDays"].rolling(3).mean()

clean_df.dropna(inplace=True)

# ===============================
# 3️⃣ FEATURES & TARGET
# ===============================

features = ["Rain_lag1", "Wet_lag1", "Rain_roll3", "Wet_roll3"]
X = clean_df[features]

targets = {
    "Rainfall": clean_df["Rainfall"],
    "WetDays": clean_df["WetDays"]
}

# ===============================
# 4️⃣ MODEL TRAINING FUNCTION
# ===============================

def train_and_evaluate(target_name, y):

    model = RandomForestRegressor(
        n_estimators=400,
        max_depth=12,
        min_samples_split=4,
        min_samples_leaf=2,
        random_state=42
    )

    tscv = TimeSeriesSplit(n_splits=5)

    r2_scores = []
    mae_scores = []

    for train_index, test_index in tscv.split(X):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        model.fit(X_train, y_train)
        predictions = model.predict(X_test)

        r2_scores.append(r2_score(y_test, predictions))
        mae_scores.append(mean_absolute_error(y_test, predictions))

    print(f"\n📊 {target_name} Performance")
    print("Average R2:", np.mean(r2_scores))
    print("Average MAE:", np.mean(mae_scores))

    # Train final model on full dataset
    model.fit(X, y)

    # Plot sorted feature importance
    importance = pd.Series(
        model.feature_importances_,
        index=features
    ).sort_values()

    plt.figure()
    importance.plot(kind="barh")
    plt.title(f"{target_name} Feature Importance")
    plt.show()

    return model


# ===============================
# 5️⃣ TRAIN MODELS
# ===============================

rain_model = train_and_evaluate("Rainfall", targets["Rainfall"])
wet_model = train_and_evaluate("WetDays", targets["WetDays"])

NameError: name 'clean_df' is not defined